# Generowanie danych do plików csv

## 1 Użyte biblioteki

In [9]:
from faker import Faker
import random
from countryinfo import CountryInfo
import csv
import string
import unicodedata
from datetime import datetime, timedelta
import os

### 1.1 Funkcja używana do generowania adresów mailowych

In [10]:
def remove_special_characters(text):
    normalized_string = unicodedata.normalize('NFD', text)
    final_string = ''.join(
        char for char in normalized_string if ord(char) <= 127 and char.isalnum() and not char.isspace()
    )
    return final_string

## 2 Generowanie informacji o ludziach z podziałem na studentów i pracowników

Generator gwarantuje unikatowość identyfikatotów, adresów email oraz numerów telefonów. Obecna implementacja umożliwia generowanie danych dla różnych krajów.

### 2.1 Klasa Person

In [11]:
class Person:
    
    student_counter = 1000
    employee_counter = 1000
    translator_counter = 100
    generated_phone_numbers = set()
    generated_emails = set()
    
    def __init__(self, position, symbol):
        self.firstNameGenerate(symbol)
        self.lastNameGenerate(symbol)
        self.status = position
        self.birthDateGenerate()
        self.cityGenerate(symbol)
        self.streetAddressGenerate(symbol)
        self.getCountryName(symbol)
        self.getStudentID()
        self.getEmployeeID()
        self.getTranslatorID()
        self.phoneNumberGenerate(symbol)
        self.emailGenerate()
        self.translator_languages()
        self.jobPosition = None
    
    def firstNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.firstName = fake.first_name()
    
    def lastNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.lastName = fake.last_name()
        
    def cityGenerate(self, symbol):
        fake = Faker(symbol)
        self.city = fake.city()
    
    def streetAddressGenerate(self, symbol):
        fake = Faker(symbol)
        self.streetAddress = fake.street_address()
    
    def getCountryName(self, symbol):
        
        country_dict = {
            "US": "Stany Zjednoczone",
            "PL": "Polska",
            "DE": "Niemcy",
            "GB": "Wielka Brytania",
            "FR": "Francja",
            "IT": "Włochy"
        }
        
        country_code = symbol.split('_')[1]
        self.country = country_dict[country_code]
    
    
    def birthDateGenerate(self):
        position = self.status
        fake = Faker()
        match position:
            case 'student':
                age_ranges = [(18, 25), (26, 30), (31, 50), (51, 65)]
                weights = [0.6, 0.2, 0.15, 0.05]
            case 'employee':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.2, 0.4, 0.3, 0.1]
            case 'translator':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.4, 0.3, 0.2, 0.1]
                
            
        selected_range = random.choices(age_ranges, weights=weights, k=1)[0]
        min_age, max_age = selected_range
        self.birthDate = fake.date_of_birth(None, min_age, max_age)
    
    def getStudentID(self):
        if self.status == 'student':
            self.studentID = Person.student_counter
            Person.student_counter += 1
    
    def getEmployeeID(self):
        if self.status == 'employee':
            self.employeeID = Person.employee_counter
            Person.employee_counter += 1
    
    def getTranslatorID(self):
        if self.status == 'translator':
            self.translatorID = Person.translator_counter
            Person.translator_counter += 1
    
    def phoneNumberGenerate(self, symbol):
        faker = Faker(symbol)
        country_code = symbol.split('_')[1]
        informations = CountryInfo(country_code)
        calling_code = informations.calling_codes()[0]
        
        while True:
            phone_number = faker.phone_number()
            if not phone_number[0] == '+':
                phone_number = "+" + calling_code + " " + phone_number
            if phone_number not in Person.generated_phone_numbers:
                self.phone = phone_number
                Person.generated_phone_numbers.add(phone_number)
                break
    
    def emailGenerate(self):
        domains = [
            "gmail.com",
            "outlook.com",
            "interia.pl",
            "yahoo.com",
            "wp.pl"
        ]
        weights = [0.4, 0.1, 0.2, 0.1, 0.2]
        trans = str.maketrans("ąćęłńóśźż", "acelnoszz")
        first_name = remove_special_characters(self.firstName)
        last_name = remove_special_characters(self.lastName)
        
        while True:
            domain =  random.choices(domains, weights=weights, k=1)[0]
            random_number = random.randint(1, 9999)
            random_separator = random.choice([".", "_", "-"])
            
            email_prefix = random.choice([
            f"{first_name}{random_number}{last_name}",
            f"{first_name}{random_separator}{last_name}",
            f"{last_name}{random_separator}{first_name}",
            f"{last_name}{random_separator}{first_name}{random_number}",
            f"{first_name}{random_number}"
            f"{last_name}{random_separator}{random_number}"
            ]).lower()
            
            email = email_prefix + "@" + domain
            if email not in Person.generated_emails:
                self.email = email
                Person.generated_emails.add(email)
                break
            
    def translator_languages(self):
        if self.status == 'translator':
            languagesIDs = [1, 2, 3, 4, 5]
            numbers_of_languages = random.randint(1, 3)
            self.languagesID = random.sample(languagesIDs, k=numbers_of_languages)

### 2.2 Przykład użycia

In [12]:

person = Person('employee', 'pl_PL')
print('First name:', person.firstName)
print('Last name:', person.lastName)
print('Position:', person.status)
print('Birth Date:', person.birthDate)
print('City:', person.city)
print('Address:', person.streetAddress)
print('Country:', person.country)
print('Employee ID:', person.employeeID)
print('Phone number:', person.phone)
print('Email:', person.email)


First name: Daniel
Last name: Tyma
Position: employee
Birth Date: 1971-09-29
City: Siedlce
Address: pl. Morcinka 258
Country: Polska
Employee ID: 1000
Phone number: +48 602 376 528
Email: tyma_daniel1124@gmail.com


### 2.3 Zapisywanie danych do pliku .csv

In [13]:
def SavetoCsv(filename, data):
    
    folder = 'data'
    os.makedirs(folder, exist_ok=True) 
    filepath = os.path.join(folder, filename)
    
    with open(filepath, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        for row in data:
            writer.writerow(row)

### 2.4 Generowanie wszystkich studentów, pracowników i tłumaczy

In [14]:
students = []
employees = []
translators = []
for i in range(1000):
    student = Person('student', 'pl_PL')
    students.append(student)
for i in range(50):
    employee = Person('employee', 'pl_PL')
    employees.append(employee)
for i in range(5):
    translator = Person('translator', 'pl_PL')
    translators.append(translator)

### 2.5 Zapisywanie danych dla studentów do csv

In [15]:
students_info = []
for i in range(len(students)):
    student = students[i]
    student_data = [
        student.studentID, 
        student.firstName, 
        student.lastName, 
        student.birthDate, 
        student.country, 
        student.city, 
        student.streetAddress, 
        student.email, 
        student.phone
    ]
    students_info.append(student_data)
SavetoCsv('student.csv', students_info)

### 2.6 Zapisywanie danych dla pracowników do csv

In [16]:
employees_info = []
for i in range(len(employees)):
    employee = employees[i]
    employee_data = [
        employee.employeeID, 
        employee.firstName, 
        employee.lastName, 
        employee.birthDate, 
        employee.country, 
        employee.city, 
        employee.streetAddress, 
        employee.email, 
        employee.phone
    ]
    employees_info.append(employee_data)
SavetoCsv('employee.csv', employees_info)

### 2.7 Zapisywanie danych dla tłumaczy do pliku

In [17]:
translators_info = []
for i in range(len(translators)):
    translator = translators[i]
    translator_data = [
        translator.translatorID, 
        translator.firstName, 
        translator.lastName, 
        translator.birthDate, 
        translator.country, 
        translator.city, 
        translator.streetAddress, 
        translator.email, 
        translator.phone
    ]
    translators_info.append(translator_data)
SavetoCsv('translator.csv', translators_info)

### 2.8 Mapa języków

In [18]:
languages = {
    1 : 'Angielski',
    2 : 'Hiszpański',
    3 : 'Francuski',
    4 : 'Niemiecki',
    5 : 'Włoski',
    6 : 'Polski'
}

### 2.9 Zapisywanie języków do csv

In [19]:
languages_info = []
for translator in translators:
    for language in translator.languagesID:
        languages_info.append([language, translator.translatorID])
SavetoCsv('translator_language.csv', languages_info)
languages_info = []
for i in range(1, 7):
    languages_info.append([i, languages[i]])
SavetoCsv('languages.csv', languages_info)

### 2.10 Generowanie stanowisk pracowników

In [20]:
def generate_positions_to_csv():
    held_positions = []
    positions = [
        'Dyrektor główny', 
        'Księgowy', 
        'Księgowy',
        'Administrator danych',
        'Pracownik administracji'
    ]
    for position in positions:
        person = random.choice(employees)
        held_positions.append([person.employeeID, position])
        employees.remove(person)
    for person in employees:
        held_positions.append([person.employeeID, 'Prowadzący zajęcia'])
    SavetoCsv('positions.csv', held_positions)
generate_positions_to_csv()

## 3 Generowanie informacji na temat lokalizacji zajęć

In [21]:
class LectureRoom:
    id_counter = 100
    lecture_rooms = set()
    
    def __init__(self):
        self.id = LectureRoom.id_counter
        self.buildingInfoGenerate()
        LectureRoom.id_counter += 1
    
    def buildingInfoGenerate(self):
        buildings = ['A-0', 'A-1', 'A-2', 'B-1', 'B-2', 'C-1', 'C-2', 'C-3']
        while True:
            floor = random.randint(1, 4)
            class_number = random.randint(1, 20)
            building = random.choice(buildings)
            new_class = str(class_number) + " " + str(floor) + building
            
            if new_class not in LectureRoom.lecture_rooms:
                self.building = building
                self.floor = floor
                self.classNumber = class_number
                LectureRoom.lecture_rooms.add(new_class)
                break

### 3.1 Przykład generowanych sal lekcyjnych

In [22]:
classes = [LectureRoom(), LectureRoom(), LectureRoom()]

for i in range(3):
    print(f"\nSala {i}")
    print('Room ID:', classes[i].id)
    print('Building:', classes[i].building)
    print('Floor:', classes[i].floor)
    print('Class number:', classes[i].classNumber)


Sala 0
Room ID: 100
Building: C-3
Floor: 1
Class number: 16

Sala 1
Room ID: 101
Building: C-2
Floor: 4
Class number: 16

Sala 2
Room ID: 102
Building: C-3
Floor: 2
Class number: 9


### 3.2 Przykładowe generowanie danych dla sal i zapisanie ich

In [23]:
rooms_info = []
rooms = []
for i in range(30):
    room = LectureRoom()
    rooms.append(room)
    room_data = [
        room.id,
        room.building, 
        room.floor, 
        room.classNumber
    ]
    rooms_info.append(room_data)
SavetoCsv('lecturerooms.csv', rooms_info)

## 4 Generowanie webinarów

W celu uniknięcia w przyszlości kolizji wynikającej z przypisania jednemu pracownikowi dwóch zajęć jednoczeńsnie, wszystkie daty wraz z ID pracownika będą zapisywane w jednym set.

In [24]:
date_employee = set()
teaching_employees = set()

In [25]:
class Webinar:
    
    webinar_counter = 1000
    generated_links = set()
    
    def __init__(self):
        self.getID()
        self.linkGenerate()
        self.priceGenerate()
        self.getTranslator()
        self.getEmployee()
        self.dateGenerate()
        self.videoLinkGenerate()
        
        
    def getID(self):
        self.id = Webinar.webinar_counter
        Webinar.webinar_counter += 1
    
    def linkGenerate(self):
        prefix = "https://teams.microsoft.com/l/meetup-join/"
        while True:
            sufix = random.choices(string.ascii_lowercase + string.digits, k = 20)
            sufix = ''.join(sufix)
            if sufix not in Webinar.generated_links:
                link = prefix + sufix
                self.link = link
                Webinar.generated_links.add(link)
                break
    
    def priceGenerate(self):
        prices = [29.99, 99, 149, 249]
        weights = [0.3, 0.5, 0.15, 0.05]
        self.price = random.choices(prices, weights=weights, k=1)[0]
    
    def getTranslator(self):
        weights = [0.2, 0.8]
        translator = random.choice(translators)
        translator = random.choices([translator, None], weights=weights, k=1)[0]
        if translator:
            languageID = random.choice(translator.languagesID)
        if translator is not None and languageID != 6:
            self.translatorID = translator.translatorID
            self.languageID = languageID
        else:
            self.translatorID = 'NULL'
            self.languageID = 6
            
    
    def getEmployee(self):
        employee = random.choice(employees)
        self.employeeID = employee.employeeID
        teaching_employees.add(employee)
    
    def dateGenerate(self):
        fake = Faker()
        teacher = self.employeeID
        hours = [
            '16:45',
            '18:30',
            '20:15'
        ]
        while True:
            hour = random.choice(hours)
            random_days = random.randint(-40, 90)    
            date = datetime.now() + timedelta(days=random_days)

            random_datetime_str = f"{date.strftime('%Y-%m-%d')} {hour}"
            random_datetime = datetime.strptime(random_datetime_str, '%Y-%m-%d %H:%M')
            
            if (random_datetime, teacher) not in date_employee:
                self.date = random_datetime
                date_employee.add((random_datetime, teacher))
                break
    
    def videoLinkGenerate(self):
        if self.date < datetime.now():
            prefix = "https://vimeo.com/"
            while True:
                sufix = random.choices(string.ascii_lowercase + string.digits, k = 40)
                sufix = ''.join(sufix)
                if sufix not in Webinar.generated_links:
                    link = prefix + sufix
                    self.videoLink = link
                    Webinar.generated_links.add(link)
                    break
        else:
            self.videoLink = 'NULL'

### 4.1 Przykładowy wygenerowany webinar

In [26]:
webinar = Webinar()
print('Webinar ID:', webinar.id)
print('Date and time:', webinar.date)
print('Employee ID:', webinar.employeeID)
print('Translator ID:', webinar.translatorID)
print('Language ID:', webinar.languageID)
print('Link:', webinar.link)
print('Video link:', webinar.videoLink)

Webinar ID: 1000
Date and time: 2025-03-16 20:15:00
Employee ID: 1008
Translator ID: NULL
Language ID: 6
Link: https://teams.microsoft.com/l/meetup-join/leis5c57hfah9v2m4cnh
Video link: NULL


### 4.2 Generowanie webinarów i zapisywanie ich do pliku

In [27]:
webinars = []
webinars_info = []
for i in range(10):
    webinar = Webinar()
    webinars.append(webinar)
    webinar_details = [
        webinar.id,
        webinar.price,
        webinar.date,
        webinar.languageID,
        webinar.translatorID,
        webinar.employeeID,
        webinar.link,
        webinar.videoLink
    ]
    webinars_info.append(webinar_details)
SavetoCsv('webinars.csv', webinars_info)

## 5 Data ważności dostępu do webinaru oraz studenci korzystający z webinarów do csv

Ponieważ dostęp do platformy z dostępem jest przez miesiąc nie musimy przejmować się kolizją uczstnictwa studentów w webinarze z ich studiami

In [28]:
def webinar_expiration_date_to_csv(students_number):
    students_webinar = set()
    result_data = []
    for i in range(students_number):
        while True:
            student = random.choice(students)
            if student not in students_webinar:
                students_webinar.add(student)
                break
        webinar = random.choice(webinars)
        if webinar.date < datetime.now():
            days_diff = random.randint(1, 30)
            expiration_date = datetime.now() + timedelta(days=days_diff)
            expiration_date = expiration_date.strftime("%Y-%m-%d")
        else:
            expiration_date = 'NULL'
        data = [webinar.id, student.studentID, expiration_date]
        result_data.append(data)
    SavetoCsv('webinar_expiration.csv', result_data)
    print(result_data[0])
webinar_expiration_date_to_csv(50)   
            

[1004, 1639, 'NULL']


# Generowanie danych dla studiów

## 1 Kierunki studiów

In [29]:
class FieldOfStudies:
    
    field_id_counter = 1
    
    def __init__(self, name, description):
        self.getID()
        self.name = name
        self.limitGenerate()
        self.entryFeeGenerate()
        self.description = description
        
    def getID(self):
        self.fieldID = FieldOfStudies.field_id_counter
        FieldOfStudies.field_id_counter += 1
    
    def limitGenerate(self):
        lower_bound = 100
        difference = random.randint(0, 5)
        self.limit = lower_bound + difference * 10
    
    def entryFeeGenerate(self):
        self.fee = random.randint(9, 18) * 10

## 2 Przedmioty

In [30]:
class Subject:
    
    subject_id_counter = 1
    
    def __init__(self, name, fieldID, description, semester):
        self.name = name
        self.fieldID = fieldID
        self.description = description
        self.getID()
        self.meetingQuantity()
        self.getEmployee()
        self.semester = semester
    
    def getID(self):
        self.id = Subject.subject_id_counter
        Subject.subject_id_counter += 1
    
    def meetingQuantity(self):
        self.quantity = random.randint(7, 18)
    
    #currently one employee could have multiple subjects and some of them have none of them
    def getEmployee(self):
        self.employeeID = random.choice(employees).employeeID

## 3 Spotkania 

In [31]:
# occupied rooms
room_and_hour = set()
# subject id & date
subject_and_date = set()

class Meeting:
    
    meeting_id_counter = 100
    
    def __init__(self, subjectID):
        self.getID()
        self.type = random.randint(1, 2)
        self.linkGenerate()
        self.subjectID = subjectID
        self.dateGenerate()
        self.GetRoomID()
        self.getTranslator()
        self.price = random.choice([29.99, 49.99, 59.99])
    
    def getID(self):
        self.id = Meeting.meeting_id_counter
        Meeting.meeting_id_counter += 1
        

    def dateGenerate(self):
        hours = [
            '10:00',
            '11:15',
            '13:00',
            '14:45'
        ]
        while True:
            hour = random.choice(hours)
            random_days = random.randint(-150, 30)    
            date = datetime.now() + timedelta(days=random_days)

            random_datetime_str = f"{date.strftime('%Y-%m-%d')} {hour}"
            random_datetime = datetime.strptime(random_datetime_str, '%Y-%m-%d %H:%M')
            
            if (random_datetime, self.subjectID) not in subject_and_date:
                self.date = random_datetime
                subject_and_date.add((random_datetime, self.subjectID))
                break
    
    def linkGenerate(self):
        if self.type == 2:
            prefix = "https://teams.microsoft.com/l/meetup-join/"
            while True:
                sufix = random.choices(string.ascii_lowercase + string.digits, k = 20)
                sufix = ''.join(sufix)
                if sufix not in Webinar.generated_links:
                    link = prefix + sufix
                    self.link = link
                    Webinar.generated_links.add(link)
                    break
        else:
            self.link = 'NULL'
    
    def GetRoomID(self):
        if self.type == 1:
            date = self.date
            while True:
                room = random.choice(rooms)
                if (room.id, date) not in room_and_hour:
                    self.roomID = room.id
                    room_and_hour.add((room.id, date))
                    break
        else:
            self.roomID = 'NULL'
    
    def getTranslator(self):
        weights = [0.2, 0.8]
        translator = random.choice(translators)
        translator = random.choices([translator, None], weights=weights, k=1)[0]
        if translator:
            languageID = random.choice(translator.languagesID)
        if translator is not None and languageID != 6:
            self.translatorID = translator.translatorID
            self.languageID = languageID
        else:
            self.translatorID = 'NULL'
            self.languageID = 6
        

## 4 Generowanie

### 4.1 Zbiory oraz początkowe dane

In [32]:
meetings = set()
subjects = set()
field_of_studies = []

it_programs = [
    {
        "name": "Informatyka ogólna",
        "description": "Kierunek obejmujący szeroką wiedzę z zakresu programowania, algorytmów oraz struktur danych.",
        "semesters": {
            1: [
                {"name": "Algorytmy i struktury danych", "description": "Nauka o algorytmach i strukturach danych, podstawy analizy algorytmów."},
                {"name": "Programowanie obiektowe", "description": "Zasady programowania w paradygmacie obiektowym, język Java lub C++."},
            ],
            2: [
                {"name": "Podstawy baz danych", "description": "Wprowadzenie do systemów baz danych, modelowanie danych, SQL."},
                {"name": "Systemy operacyjne", "description": "Podstawy działania systemów operacyjnych, zarządzanie pamięcią, procesami."},
            ],
            3: [
                {"name": "Sieci komputerowe", "description": "Wprowadzenie do sieci komputerowych, protokoły, topologie, bezpieczeństwo sieci."},
            ]
        }
    },
    {
        "name": "Inżynieria oprogramowania",
        "description": "Skupia się na tworzeniu, testowaniu i zarządzaniu projektami oprogramowania.",
        "semesters": {
            1: [
                {"name": "Projektowanie systemów informatycznych", "description": "Projektowanie i analiza systemów informatycznych, UML, diagramy."},
                {"name": "Testowanie oprogramowania", "description": "Metody testowania oprogramowania, testowanie jednostkowe, integracyjne, automatyczne."},
            ],
            2: [
                {"name": "Zarządzanie projektem IT", "description": "Metodyki zarządzania projektami IT, Agile, Scrum, Kanban."},
                {"name": "Bazy danych", "description": "Zaawansowane techniki pracy z bazami danych, normalizacja, transakcje."},
            ],
            3: [
                {"name": "Programowanie w języku Python", "description": "Podstawy programowania w języku Python, struktury danych, programowanie obiektowe."},
            ]
        }
    },
    {
        "name": "Sztuczna inteligencja",
        "description": "Kierunek związany z tworzeniem systemów inteligentnych, w tym rozwiązań z zakresu uczenia maszynowego.",
        "semesters": {
            1: [
                {"name": "Uczenie maszynowe", "description": "Metody uczenia maszynowego, algorytmy nadzorowane i nienadzorowane."},
                {"name": "Sieci neuronowe", "description": "Teoria i praktyka sieci neuronowych, ich zastosowanie w rozwiązywaniu problemów."},
            ],
            2: [
                {"name": "Analiza danych", "description": "Zbieranie, przetwarzanie i analiza danych przy użyciu narzędzi analitycznych."},
                {"name": "Algorytmy genetyczne", "description": "Wykorzystanie algorytmów inspirowanych ewolucją do rozwiązywania problemów optymalizacyjnych."},
            ],
            3: [
                {"name": "Rozpoznawanie obrazów", "description": "Przetwarzanie i analiza obrazów, rozpoznawanie wzorców przy użyciu sztucznej inteligencji."},
            ]
        }
    },
    {
        "name": "Cyberbezpieczeństwo",
        "description": "Skupia się na ochronie systemów komputerowych i sieci przed zagrożeniami z sieci.",
        "semesters": {
            1: [
                {"name": "Podstawy kryptografii", "description": "Teoria kryptografii, szyfrowanie, algorytmy kryptograficzne."},
                {"name": "Bezpieczeństwo systemów operacyjnych", "description": "Zabezpieczanie systemów operacyjnych przed atakami, zarządzanie uprawnieniami."},
            ],
            2: [
                {"name": "Analiza zagrożeń w sieciach komputerowych", "description": "Metody identyfikowania i eliminowania zagrożeń w sieciach komputerowych."},
                {"name": "Technologie ochrony danych", "description": "Zabezpieczanie danych przed nieautoryzowanym dostępem, zarządzanie prywatnością."},
            ],
            3: [
                {"name": "Ataki i obrona w cyberprzestrzeni", "description": "Techniki ataków i obrony przed nimi, analiza ataków DDoS, phishing."},
            ]
        }
    },
    {
        "name": "Big Data",
        "description": "Kierunek poświęcony analizie ogromnych zbiorów danych oraz wykorzystywaniu technologii do ich przetwarzania.",
        "semesters": {
            1: [
                {"name": "Przetwarzanie danych w chmurze", "description": "Wykorzystanie technologii chmurowych do przechowywania i przetwarzania dużych zbiorów danych."},
                {"name": "Hadoop i Spark", "description": "Platformy do przetwarzania dużych danych, Hadoop, Spark, MapReduce."},
            ],
            2: [
                {"name": "Analiza dużych zbiorów danych", "description": "Techniki analizy i przetwarzania dużych zbiorów danych, analiza statystyczna."},
                {"name": "Bazy danych NoSQL", "description": "Bazy danych NoSQL, ich struktura i zastosowania w analizie dużych danych."},
            ],
            3: [
                {"name": "Systemy rekomendacyjne", "description": "Tworzenie systemów rekomendacyjnych, algorytmy uczenia maszynowego w rekomendacjach."},
            ]
        }
    }
]


### 4.2 Generowanie kierunków i przedmiotów oraz spotkań

In [33]:
for program in it_programs:

    field = FieldOfStudies(program['name'], program['description'])
    field_of_studies.append(field)
    
    for semester, subjects_list in program['semesters'].items():
        for subject in subjects_list:
            new_subject = Subject(subject['name'], field.fieldID, subject['description'], semester)
            subjects.add(new_subject)
            
            for i in range(new_subject.quantity):
                meeting = Meeting(new_subject.id)
                meetings.add(meeting)

### 4.3 Zapisywanie do pliku csv

In [34]:
subjects_info = []
field_of_studies_info = []
meetings_info = []

for meeting in meetings:
    meetings_info.append([meeting.id, meeting.type, meeting.subjectID, meeting.date, meeting.link, meeting.roomID, meeting.languageID, meeting.translatorID, meeting.price])
SavetoCsv('Meeting.csv', meetings_info)

for subject in subjects:
    subjects_info.append([subject.id, subject.fieldID, subject.name, subject.description, subject.quantity, subject.employeeID, subject.semester])
SavetoCsv('Subjects.csv', subjects_info)

for field in field_of_studies:
    field_of_studies_info.append([field.fieldID, field.name, field.description, field.limit, field.fee])
SavetoCsv('Fields.csv', field_of_studies_info)



# Pozostałe tabele w studiach

In [35]:
SavetoCsv('Meetingtype.csv', [[1, 'zajęcia w formie stacjonarnej'], [2, 'zajęcia w formie zdalnej']])

## 1 Lista studentów dla każdego wydziału

In [36]:
field_semester_students = {field.fieldID: {semester: [] for semester in range(1, 4)} for field in field_of_studies}

def students_list():
    faculty_student_list = []
    used_students = set()

    student_lists_by_semester = [[] for _ in range(len(field_of_studies))]

    for i, field in enumerate(field_of_studies):
        capacity_per_semester = field.limit
        diff = random.randint(85, 100)

        for semester in range(3):
            allocated_students = 0
            while allocated_students < capacity_per_semester - diff:
                student = random.choice(students)
                if student not in used_students:
                    used_students.add(student)
                    student_lists_by_semester[i].append((semester + 1, student.studentID))
                    field_semester_students[field.fieldID][semester + 1].append(student.studentID)
                    allocated_students += 1

    for i, field in enumerate(field_of_studies):
        field_id = field.fieldID
        for semester, studentID in student_lists_by_semester[i]:
            faculty_student_list.append([field_id, studentID, semester])

    SavetoCsv('FacultyStudentList.csv', faculty_student_list)

students_list()


## 2 Oceny za przedmioty

In [37]:
low_attendance_students_ID = set()

def grades():
    grade_list = []
    weights = [0.3, 0.1, 0.25, 0.15, 0.1]
    grades = ['NULL', 2, 3, 4, 5]

    for field in field_of_studies:
        fieldID = field.fieldID

        for semester in range(1, 4):
            for studentID in field_semester_students[fieldID][semester]:

                for subject in subjects:

                    if subject.fieldID == fieldID:
                        if subject.semester == semester:

                            grade = random.choices(grades, weights=weights, k=1)[0]
                            if grade == 'NULL':
                                low_attendance_students_ID.add((studentID, subject))
                            else:
                                grade_list.append([subject.id, studentID, grade])

    SavetoCsv('GradeList.csv', grade_list)

grades()
